# Exercise 2: LangChain ReAct Agent as a NAT Function

In this exercise you'll learn how to:
1. Understand a pre-built LangChain ReAct agent
2. Register it as a **NeMo Agent Toolkit Function**
3. Run it through NAT's orchestration layer

## Why register agents as NAT Functions?

- **Composability** - mix agents from different frameworks in one workflow
- **Observability** - automatic tracing and profiling (Part 3)
- **Unified config** - manage everything via YAML
- **Deployment** - `nat serve` deploys the full workflow as an API

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Step 1: Examine the LangChain Agent Code

The implementation is in `workshop_agents/langchain_research_agent.py`. Key parts:

1. **Config class** inherits from `FunctionBaseConfig` with `name="langchain_research_agent"`
2. **`@register_function` decorator** - Tells NAT this is a composable function
3. **Builder pattern** - `_builder.get_llm()` and `_builder.get_tools()` retrieve instrumented components
4. **`langchain.agents.create_agent()`** - New LangGraph-based agent API
5. **`FunctionInfo.from_fn()`** - Registers the callable with NAT

In [2]:
with open("../../workshop_agents/langchain_research_agent.py") as f:
    print(f.read())

"""
LangChain Research Agent - registered as a NeMo Agent Toolkit Function.

Uses langchain.agents.create_agent (LangGraph-based) to build a research
agent that can search Wikipedia. Wrapped as a NAT Function for composition.
"""

from typing import List
from pydantic import Field

from nat.cli.register_workflow import register_function
from nat.builder.function_info import FunctionInfo
from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.data_models.function import FunctionBaseConfig
from nat.data_models.component_ref import FunctionRef, LLMRef


class LangChainResearchAgentConfig(FunctionBaseConfig, name="langchain_research_agent"):
    """Configuration for the LangChain research agent."""

    llm_ref: LLMRef = Field(description="Reference to the LLM to use")
    tools_ref: List[FunctionRef] = Field(description="References to tools the agent can use")
    description: str = Field(default="Agent badawczy korzystający z LangChain")
 

## Configuration Object

Each NeMo Agent Toolkit **function** requires a configuration object that inherits from **FunctionBaseConfig**. The **FunctionBaseConfig** class and ultimately all NeMo Agent Toolkit configuration objects are subclasses of the **pydantic.BaseModel** class from the Pydantic Library, which provides a way to define and validate configuration objects. Each configuration object defines the parameters used to create runtime instances of functions (or other component type), each with different functionality based on configuration settings. 

## Function Registration

NeMo Agent Toolkit relies on a configuration with builder pattern to define most components. For functions, `@register_function` is a decorator that must be specified to inform the toolkit that a function should be accessible automatically by name when referenced. The decorator requires that a config_type is specified. This is done to ensure type safety and validation.

The parameters to the decorated function are always:

- the configuration type of the function component (FunctionBaseConfig)
- a Builder which can be used to dynamically query and get other workflow components (Builder)

## Elements of a function

1) the definition of a closure function `research` which captures the instantiated agent executor and uses that to invoke the agent and return the response. And 2) the use of the @register_function decorator

## Step 2: Examine the YAML Config

Notice how `config.yaml` references our custom function type `langchain_research_agent`
and wires it to the Bielik LLM and tools.

In [3]:
with open("config.yaml") as f:
    print(f.read())

# =============================================================
# Exercise 2: LangChain ReAct Agent as NAT Function
# =============================================================
# This config registers a LangChain ReAct research agent
# as a NeMo Agent Toolkit Function, making it composable
# with other agents in a unified workflow.
# =============================================================

llms:
  bielik:
    _type: openai
    base_url: "${VLLM_BASE_URL}"
    model: "${VLLM_MODEL_NAME}"
    api_key: "${VLLM_API_KEY}"
    temperature: 0.7
    max_tokens: 2048

functions:
  # Built-in tools that the LangChain agent will use
  wiki_search:
    _type: wiki_search
    max_results: 2

  current_time:
    _type: current_datetime

  # Our custom LangChain agent registered as a NAT Function
  research_agent:
    _type: langchain_research_agent
    llm_ref: bielik
    tools_ref:
      - wiki_search
      - current_time
    description: "Agent badawczy wyszukujący i podsumowujący informa

## Step 3: Run the LangChain Agent via NAT

In [4]:
!nat run --config_file config.yaml --input "Zbadaj historię sztucznej inteligencji w Polsce."

2026-04-15 17:46:32 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-15 17:46:32 - ERROR    - nat.data_models.config:118 - Requested functions type `langchain_research_agent` not found. Have you ensured the necessary package has been installed with `uv pip install`?
Available functions names:
 - nat.plugins.langchain/langgraph_wrapper
 - nat.plugins.langchain.tools/code_generation
 - nat.plugins.langchain.tools/tavily_internet_search
 - nat.plugins.langchain.tools/wiki_search
 - nat.plugins.langchain.agent.auto_memory_wrapper/auto_memory_agent
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_init
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_recombiner
 - nat.plugins.langchain.agent.react_agent/react_agent
 - nat.plugins.langchain.agent.react_agent/per_user_react_agent
 - nat.plugins.langchain.agent.reasoning_agent/reasoning_agent
 - nat.plugins.langchain.agent.responses_api_agent/responses_api_agent
 - nat.plugins.langchain.

## Key Pattern: `@register_function`

```python
from nat.cli.register_workflow import register_function
from nat.builder.function_info import FunctionInfo
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.data_models.function import FunctionBaseConfig

class MyConfig(FunctionBaseConfig, name="my_agent_type"):
    llm_ref: LLMRef = Field(...)

@register_function(config_type=MyConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def my_agent(_config, _builder):
    llm = await _builder.get_llm(...)     # instrumented automatically
    tools = await _builder.get_tools(...)  # instrumented automatically
    
    # ... build your agent with langchain.agents.create_agent() ...
    
    async def research(question: str) -> str:
        result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
        return result["messages"][-1].content
    
    yield FunctionInfo.from_fn(research, description="...")  # Register with NAT
```

**Next:** We'll do the same with a CrewAI agent crew.